In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

# -------------------------
# Define DeepGlobe RGB class colors
# -------------------------
CLASS_COLORS = {
     1: (166, 202, 240),   # airplane
    2: (128, 128, 0),     # bare soil
    3: (0, 0, 128),       # buildings
    4: (255, 0, 0),       # cars
    5: (0, 128, 0),       # chaparral
    6: (128, 0, 0),       # court
    7: (255, 233, 233),   # dock
    8: (160, 160, 164),   # field
    9: (0, 128, 128),     # grass
    10: (90, 87, 255),    # mobile home
    11: (255, 255, 0),    # pavement
    12: (255, 192, 0),    # sand
    13: (0, 0, 255),      # sea
    14: (255, 0, 92),     # ship
    15: (128, 0, 128),    # tanks
    16: (0, 255, 0),      # trees
    17: (0, 255, 255),    # water
}

# Reverse the color mapping: RGB tuple → class index
COLOR_TO_CLASS = {v: k for k, v in CLASS_COLORS.items()}

# -------------------------
# Set your paths
# -------------------------
images_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_images"
masks_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_masks"  # RGB masks
output_csv_path = "labels.csv"

# -------------------------
# Process images
# -------------------------
data = []
image_files = sorted(os.listdir(images_folder))

for fname in tqdm(image_files, desc="Extracting labels from RGB masks"):
    mask_path = os.path.join(masks_folder, fname)
    
    if not os.path.exists(mask_path):
        print(f"❌ Mask not found for {fname}, skipping...")
        continue

    # Read RGB mask
    mask_rgb = cv2.imread(mask_path)
    if mask_rgb is None:
        print(f"⚠️ Could not read mask for {fname}")
        continue

    mask_rgb = cv2.cvtColor(mask_rgb, cv2.COLOR_BGR2RGB)

    # Find all unique RGB colors in the mask
    unique_colors = np.unique(mask_rgb.reshape(-1, 3), axis=0)

    # Map present colors to class indices
    class_indices = []
    for color in map(tuple, unique_colors):
        if color in COLOR_TO_CLASS:
            class_indices.append(COLOR_TO_CLASS[color])

    if len(class_indices) == 0:
        continue

    label_str = " ".join(map(str, sorted(class_indices)))
    data.append({"filename": fname, "labels": label_str})

# -------------------------
# Save to CSV
# -------------------------
df = pd.DataFrame(data)
df.to_csv(output_csv_path, index=False)
print(f"\n✅ CSV saved to: {output_csv_path}")


Extracting labels from RGB masks: 100%|███████| 630/630 [00:30<00:00, 20.68it/s]


✅ CSV saved to: labels.csv


In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

# -------------------------
# Paths and Settings
# -------------------------
main_path ="/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/"
images_folder = os.path.join(main_path, "train_images")
output_folder = os.path.join(main_path, "SAM_outputs")
labels_csv = os.path.join(main_path, "labels.csv")

os.makedirs(output_folder, exist_ok=True)

# -------------------------
# Load labels.csv
# -------------------------
df = pd.read_csv(labels_csv)

# image_to_classes: {'image_001.png': [1, 3, 5], ...}
image_to_classes = {
    row["filename"]: list(map(int, row["labels"].split()))
    for _, row in df.iterrows()
}

# -------------------------
# Initialize SAM
# -------------------------
sam_checkpoint = "/home/iiitdmk-param/AMB/Proposed_3/SAM/sam_vit_h_4b8939.pth"
model_type = "vit_h"
device = "cuda"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)
mask_generator = SamAutomaticMaskGenerator(sam)

# -------------------------
# Helper: Overlay SAM masks
# -------------------------
def overlay_sam_masks(image, anns):
    if len(anns) == 0:
        return image
    sorted_anns = sorted(anns, key=lambda x: x['area'], reverse=True)
    mask_overlay = np.ones((image.shape[0], image.shape[1], 4))  # RGBA
    mask_overlay[:, :, 3] = 0  # Alpha
    for ann in sorted_anns:
        m = ann['segmentation']
        color_mask = np.concatenate([np.random.random(3), [0.35]])
        mask_overlay[m] = color_mask
    return mask_overlay

# -------------------------
# Process Each Image Once
# -------------------------
print("🚀 Generating SAM overlays for all images...")

for fname in tqdm(image_to_classes.keys(), desc="Processing images"):
    img_path = os.path.join(images_folder, fname)
    image = cv2.imread(img_path)
    if image is None:
        print(f"⚠️ Skipping unreadable image: {fname}")
        continue
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Generate SAM masks
    masks = mask_generator.generate(image)

    # Create overlay visualization
    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    overlay = overlay_sam_masks(image, masks)
    plt.imshow(overlay)
    plt.axis('off')

    # Save to shared SAM_outputs/ folder
    save_path = os.path.join(output_folder, fname)
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
    plt.close()

print("\n✅ All SAM overlays saved to:", output_folder)


/home/iiitdmk-param/AMB/Proposed_3/SAM/segment-anything-main/segment_anything/build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


🚀 Generating SAM overlays for all images...


Processing images: 100%|██████████████████████| 630/630 [26:05<00:00,  2.49s/it]


✅ All SAM overlays saved to: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_outputs


In [ ]:
import os
import numpy as np
from skimage.segmentation import find_boundaries
from PIL import Image

def rgb_to_label_mask(rgb_image):
    """Convert RGB segmentation image to label mask."""
    rgb_array = np.array(rgb_image)
    if rgb_array.ndim == 3 and rgb_array.shape[2] == 3:
        integer_image = (rgb_array[:, :, 0].astype(np.int64) << 16) + \
                       (rgb_array[:, :, 1].astype(np.int64) << 8) + \
                       (rgb_array[:, :, 2].astype(np.int64))
        
        unique_colors = np.unique(integer_image)
        label_mask = np.zeros_like(integer_image, dtype=np.int32)
        
        for label_id, color in enumerate(unique_colors, start=1):
            label_mask[integer_image == color] = label_id
            
        return label_mask
    else:
        return rgb_array.astype(np.int32)

def get_color_groups(point_rgb_image):
    """Extract point groups by color from point annotation image."""
    point_array = np.array(point_rgb_image)
    color_groups = {}
    
    if point_array.ndim == 3 and point_array.shape[2] == 3:
        unique_colors = set()
        height, width, _ = point_array.shape
        
        for y in range(height):
            for x in range(width):
                r, g, b = point_array[y, x]
                if (r, g, b) != (0, 0, 0):
                    unique_colors.add((r, g, b))
        
        for color in unique_colors:
            r, g, b = color
            mask = (point_array[:, :, 0] == r) & \
                   (point_array[:, :, 1] == g) & \
                   (point_array[:, :, 2] == b)
            color_groups[color] = mask.astype(np.uint8)
            
    return color_groups

def process_single_image(sam_image_path, point_image_path, output_dir):
    """Process a single image pair with resizing."""
    # Load images
    sam_image = Image.open(sam_image_path).convert('RGB')
    point_image = Image.open(point_image_path).convert('RGB')
    
    # Get SAM image dimensions
    sam_width, sam_height = sam_image.size
    
    # Resize point image to match SAM image dimensions
    point_image = point_image.resize((sam_width, sam_height), Image.NEAREST)
    
    # Convert SAM image to label mask
    sam_mask = rgb_to_label_mask(sam_image)
    color_groups = get_color_groups(point_image)
    
    if not color_groups:
        print(f"No point annotations found in {point_image_path}")
        return None, None
    
    height, width = sam_mask.shape
    instance_mask = np.zeros((height, width), dtype=np.int32)
    current_instance_id = 1
    
    for color, point_mask in color_groups.items():
        segments_with_points = np.unique(sam_mask[point_mask > 0])
        segments_with_points = segments_with_points[segments_with_points > 0]
        
        if len(segments_with_points) > 0:
            merge_mask = np.isin(sam_mask, segments_with_points)
            instance_mask[merge_mask] = current_instance_id
            current_instance_id += 1
    
    boundaries = find_boundaries(instance_mask, mode='outer', background=0)
    binary_edges = boundaries.astype(np.uint8) * 255
    
    base_name = os.path.splitext(os.path.basename(sam_image_path))[0]
    
    edge_image = Image.fromarray(binary_edges)
    edge_output_path = os.path.join(output_dir, f"{base_name}_edges.png")
    edge_image.save(edge_output_path)
    
    print(f"Processed {base_name}: {len(color_groups)} color groups, {current_instance_id-1} instances")
    
    return binary_edges, instance_mask

def process_folders(sam_folder, point_folder, output_folder):
    """Process all images in folders."""
    os.makedirs(output_folder, exist_ok=True)
    
    sam_files = sorted([f for f in os.listdir(sam_folder) if f.endswith(('.png', '.jpg', '.jpeg'))])
    point_files = sorted([f for f in os.listdir(point_folder) if f.endswith(('.png', '.jpg', '.jpeg'))])
    
    print(f"Found {len(sam_files)} SAM files and {len(point_files)} point annotation files")
    
    successful = 0
    failed = 0
    
    for sam_file in sam_files:
        point_file = sam_file
        if point_file not in point_files:
            print(f"Warning: No point annotation found for {sam_file}, skipping...")
            failed += 1
            continue
        
        sam_path = os.path.join(sam_folder, sam_file)
        point_path = os.path.join(point_folder, point_file)
        
        try:
            process_single_image(sam_path, point_path, output_folder)
            successful += 1
        except Exception as e:
            print(f"Error processing {sam_file}: {e}")
            failed += 1
    
    print(f"\nProcessing complete! Successful: {successful}, Failed: {failed}")

# === JUPYTER NOTEBOOK USAGE ===
# Set your paths here:
sam_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_outputs"  # CHANGE THIS
point_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_point_masks_all_regions_1"  # CHANGE THIS  
output_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_outputs_edge_maps" # CHANGE THIS

# Run the processing
process_folders(sam_folder, point_folder, output_folder)

In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import time
from skimage.segmentation import find_boundaries

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

def rgb_to_label_mask_gpu(rgb_image):
    """Convert RGB segmentation image to label mask on GPU."""
    rgb_tensor = torch.from_numpy(np.array(rgb_image)).to(device)
    
    if rgb_tensor.dim() == 3 and rgb_tensor.shape[2] == 3:
        # Convert RGB to single integer values
        integer_image = (rgb_tensor[:, :, 0].long() << 16) + \
                       (rgb_tensor[:, :, 1].long() << 8) + \
                       (rgb_tensor[:, :, 2].long())
        
        # Get unique colors and create mapping
        unique_colors = torch.unique(integer_image)
        label_mask = torch.zeros_like(integer_image, dtype=torch.int32, device=device)
        
        for label_id, color in enumerate(unique_colors, start=1):
            label_mask[integer_image == color] = label_id
            
        return label_mask.cpu().numpy()  # Return to CPU for compatibility with other functions
    else:
        return rgb_tensor.cpu().numpy().astype(np.int32)

def get_color_groups_gpu(point_rgb_image):
    """Extract point groups by color from point annotation image using GPU."""
    point_tensor = torch.from_numpy(np.array(point_rgb_image)).to(device)
    color_groups = {}
    
    if point_tensor.dim() == 3 and point_tensor.shape[2] == 3:
        # Get unique colors (excluding black)
        unique_colors = torch.unique(point_tensor.view(-1, 3), dim=0)
        unique_colors = unique_colors[~torch.all(unique_colors == 0, dim=1)]  # Remove black
        
        for color in unique_colors:
            r, g, b = color
            mask = (point_tensor[:, :, 0] == r) & \
                   (point_tensor[:, :, 1] == g) & \
                   (point_tensor[:, :, 2] == b)
            color_groups[tuple(color.cpu().numpy())] = mask.cpu().numpy().astype(np.uint8)
            
    return color_groups

def process_single_image_gpu(sam_image_path, point_image_path, output_dir, target_size=256):
    """Process a single image pair - resize SAM to match point annotation size."""
    try:
        # Load images
        sam_image = Image.open(sam_image_path).convert('RGB')
        point_image = Image.open(point_image_path).convert('RGB')
        
        print(f"  Original SAM size: {sam_image.size}, Point size: {point_image.size}")
        
        # Resize SAM image to match point annotation size (256x256)
        if sam_image.size != (target_size, target_size):
            print(f"  Resizing SAM from {sam_image.size} to ({target_size}, {target_size})")
            sam_image = sam_image.resize((target_size, target_size), Image.NEAREST)
        
        # Convert SAM image to label mask on GPU
        sam_mask = rgb_to_label_mask_gpu(sam_image)
        color_groups = get_color_groups_gpu(point_image)
        
        print(f"  Found {len(color_groups)} color groups in point annotations")
        
        if not color_groups:
            print(f"  No point annotations found in {point_image_path}")
            return None, None
        
        # Move sam_mask to GPU for faster processing
        sam_mask_tensor = torch.from_numpy(sam_mask).to(device)
        height, width = sam_mask.shape
        
        instance_mask = np.zeros((height, width), dtype=np.int32)
        current_instance_id = 1
        
        for color, point_mask in color_groups.items():
            # Move point mask to GPU
            point_mask_tensor = torch.from_numpy(point_mask).to(device)
            
            # Find segments with points on GPU
            masked_sam = sam_mask_tensor[point_mask_tensor > 0]
            segments_with_points = torch.unique(masked_sam)
            segments_with_points = segments_with_points[segments_with_points > 0]
            
            if len(segments_with_points) > 0:
                # Create merge mask on GPU
                merge_mask = torch.isin(sam_mask_tensor, segments_with_points)
                instance_mask[merge_mask.cpu().numpy()] = current_instance_id
                current_instance_id += 1
        
        # Find boundaries
        boundaries = find_boundaries(instance_mask, mode='outer', background=0)
        binary_edges = boundaries.astype(np.uint8) * 255
        
        base_name = os.path.splitext(os.path.basename(sam_image_path))[0]
        
        edge_image = Image.fromarray(binary_edges)
        edge_output_path = os.path.join(output_dir, f"{base_name}_edges.png")
        edge_image.save(edge_output_path)
        
        print(f"  Created {current_instance_id-1} instances, saved to {edge_output_path}")
        
        # Clear GPU memory
        torch.cuda.empty_cache()
        
        return binary_edges, instance_mask
        
    except Exception as e:
        print(f"  Error in process_single_image_gpu: {e}")
        import traceback
        traceback.print_exc()
        torch.cuda.empty_cache()
        return None, None

def process_folders_gpu(sam_folder, point_folder, output_folder, target_size=256):
    """Process all images in folders with GPU acceleration."""
    # Check if folders exist
    if not os.path.exists(sam_folder):
        print(f"Error: SAM folder '{sam_folder}' does not exist!")
        return
    
    if not os.path.exists(point_folder):
        print(f"Error: Point folder '{point_folder}' does not exist!")
        return
    
    # Create output directory
    os.makedirs(output_folder, exist_ok=True)
    print(f"Output folder: {output_folder}")
    print(f"Target size: {target_size}x{target_size}")
    
    # Get file lists
    sam_files = sorted([f for f in os.listdir(sam_folder) if f.endswith(('.png', '.jpg', '.jpeg'))])
    point_files = sorted([f for f in os.listdir(point_folder) if f.endswith(('.png', '.jpg', '.jpeg'))])
    
    print(f"Found {len(sam_files)} SAM files and {len(point_files)} point annotation files")
    
    if not sam_files:
        print("No SAM files found!")
        return
    
    if not point_files:
        print("No point annotation files found!")
        return
    
    successful = 0
    failed = 0
    skipped = 0
    
    start_time = time.time()
    
    # Process each file
    for i, sam_file in enumerate(sam_files):
        print(f"\nProcessing {i+1}/{len(sam_files)}: {sam_file}")
        
        point_file = sam_file
        if point_file not in point_files:
            print(f"  Warning: No point annotation found for {sam_file}, skipping...")
            skipped += 1
            continue
        
        sam_path = os.path.join(sam_folder, sam_file)
        point_path = os.path.join(point_folder, point_file)
        
        # Check if files exist
        if not os.path.exists(sam_path):
            print(f"  Error: SAM file {sam_path} does not exist!")
            failed += 1
            continue
            
        if not os.path.exists(point_path):
            print(f"  Error: Point file {point_path} does not exist!")
            failed += 1
            continue
        
        try:
            result = process_single_image_gpu(sam_path, point_path, output_folder, target_size)
            if result[0] is not None:
                successful += 1
            else:
                failed += 1
        except Exception as e:
            print(f"  Error processing {sam_file}: {e}")
            failed += 1
            
        # Print progress every 10 images
        if (i + 1) % 10 == 0:
            elapsed = time.time() - start_time
            images_per_second = (i + 1) / elapsed
            print(f"  Processed {i+1}/{len(sam_files)} images in {elapsed:.1f}s ({images_per_second:.1f} img/s)")
    
    end_time = time.time()
    total_time = end_time - start_time
    
    print(f"\n{'='*50}")
    print(f"GPU PROCESSING COMPLETE!")
    print(f"{'='*50}")
    print(f"Total time: {total_time:.2f} seconds")
    print(f"Images processed: {len(sam_files)}")
    print(f"Successful: {successful}")
    print(f"Failed: {failed}")
    print(f"Skipped: {skipped}")
    print(f"Average speed: {len(sam_files)/total_time:.1f} images/second")
    print(f"Output folder: {output_folder}")
    
    # Show output files
    output_files = [f for f in os.listdir(output_folder) if f.endswith('_edges.png')]
    if output_files:
        print(f"\nGenerated {len(output_files)} edge maps:")
        for f in output_files[:5]:
            print(f"  {f}")
        if len(output_files) > 5:
            print(f"  ... and {len(output_files) - 5} more")
    else:
        print(f"\nNo output files found in {output_folder}")

# === JUPYTER NOTEBOOK USAGE ===
# Set your paths here (USE ABSOLUTE PATHS):
sam_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_outputs"  # CHANGE THIS
point_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_point_masks_all_regions_1"  # CHANGE THIS  
output_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_outputs_edge_maps" # CHANGE THIS
target_size = 256  # Point annotation size

print("Starting GPU processing...")
print(f"SAM folder: {sam_folder}")
print(f"Point folder: {point_folder}")
print(f"Output folder: {output_folder}")
print(f"Resizing SAM outputs to: {target_size}x{target_size}")

# Run the GPU processing
process_folders_gpu(sam_folder, point_folder, output_folder, target_size)

print("\nGPU processing completed!")

In [4]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import time
from skimage.segmentation import find_boundaries

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Define background color to ignore
BACKGROUND_COLOR = (100, 100, 100)  # Gray background

def rgb_to_label_mask_gpu(rgb_image):
    """Convert RGB segmentation image to label mask on GPU."""
    rgb_tensor = torch.from_numpy(np.array(rgb_image)).to(device)
    
    if rgb_tensor.dim() == 3 and rgb_tensor.shape[2] == 3:
        # Convert RGB to single integer values
        integer_image = (rgb_tensor[:, :, 0].long() << 16) + \
                       (rgb_tensor[:, :, 1].long() << 8) + \
                       (rgb_tensor[:, :, 2].long())
        
        # Get unique colors and create mapping
        unique_colors = torch.unique(integer_image)
        label_mask = torch.zeros_like(integer_image, dtype=torch.int32, device=device)
        
        for label_id, color in enumerate(unique_colors, start=1):
            label_mask[integer_image == color] = label_id
            
        return label_mask.cpu().numpy()
    else:
        return rgb_tensor.cpu().numpy().astype(np.int32)

def get_color_groups_gpu(point_rgb_image):
    """Extract point groups by color from point annotation image using GPU, ignoring background."""
    point_tensor = torch.from_numpy(np.array(point_rgb_image)).to(device)
    color_groups = {}
    
    if point_tensor.dim() == 3 and point_tensor.shape[2] == 3:
        # Get unique colors (excluding the specific background color)
        unique_colors = torch.unique(point_tensor.view(-1, 3), dim=0)
        
        # Convert background color to tensor for comparison
        bg_color = torch.tensor(BACKGROUND_COLOR, device=device, dtype=point_tensor.dtype)
        
        # Remove background color and any other colors you want to ignore
        mask = ~torch.all(unique_colors == bg_color, dim=1)
        unique_colors = unique_colors[mask]
        
        # Also remove pure black if it exists (additional safety)
        black_mask = ~torch.all(unique_colors == 0, dim=1)
        unique_colors = unique_colors[black_mask]
        
        for color in unique_colors:
            r, g, b = color
            mask = (point_tensor[:, :, 0] == r) & \
                   (point_tensor[:, :, 1] == g) & \
                   (point_tensor[:, :, 2] == b)
            color_groups[tuple(color.cpu().numpy())] = mask.cpu().numpy().astype(np.uint8)
            
    return color_groups

def is_background_pixel(pixel, bg_color=BACKGROUND_COLOR):
    """Check if a pixel is the background color."""
    return tuple(pixel) == bg_color

def process_single_image_gpu(sam_image_path, point_image_path, output_dir, target_size=256):
    """Process a single image pair - resize SAM to match point annotation size."""
    try:
        # Load images
        sam_image = Image.open(sam_image_path).convert('RGB')
        point_image = Image.open(point_image_path).convert('RGB')
        
        print(f"  Original SAM size: {sam_image.size}, Point size: {point_image.size}")
        
        # Resize SAM image to match point annotation size (256x256)
        if sam_image.size != (target_size, target_size):
            print(f"  Resizing SAM from {sam_image.size} to ({target_size}, {target_size})")
            sam_image = sam_image.resize((target_size, target_size), Image.NEAREST)
        
        # Convert SAM image to label mask on GPU
        sam_mask = rgb_to_label_mask_gpu(sam_image)
        color_groups = get_color_groups_gpu(point_image)
        
        print(f"  Found {len(color_groups)} color groups in point annotations (excluding background)")
        
        # Debug: Show the colors found
        if color_groups:
            print(f"  Colors found: {list(color_groups.keys())}")
        
        if not color_groups:
            print(f"  No point annotations found in {point_image_path} (only background)")
            return None, None
        
        # Move sam_mask to GPU for faster processing
        sam_mask_tensor = torch.from_numpy(sam_mask).to(device)
        height, width = sam_mask.shape
        
        instance_mask = np.zeros((height, width), dtype=np.int32)
        current_instance_id = 1
        
        for color, point_mask in color_groups.items():
            # Move point mask to GPU
            point_mask_tensor = torch.from_numpy(point_mask).to(device)
            
            # Find segments with points on GPU
            masked_sam = sam_mask_tensor[point_mask_tensor > 0]
            segments_with_points = torch.unique(masked_sam)
            segments_with_points = segments_with_points[segments_with_points > 0]
            
            if len(segments_with_points) > 0:
                # Create merge mask on GPU
                merge_mask = torch.isin(sam_mask_tensor, segments_with_points)
                instance_mask[merge_mask.cpu().numpy()] = current_instance_id
                current_instance_id += 1
                print(f"    Color {color}: merged {len(segments_with_points)} segments into instance {current_instance_id-1}")
            else:
                print(f"    Color {color}: no SAM segments found with points")
        
        # Find boundaries
        boundaries = find_boundaries(instance_mask, mode='outer', background=0)
        binary_edges = boundaries.astype(np.uint8) * 255
        
        base_name = os.path.splitext(os.path.basename(sam_image_path))[0]
        
        edge_image = Image.fromarray(binary_edges)
        edge_output_path = os.path.join(output_dir, f"{base_name}_edges.png")
        edge_image.save(edge_output_path)
        
        print(f"  Created {current_instance_id-1} instances, saved to {edge_output_path}")
        
        # Clear GPU memory
        torch.cuda.empty_cache()
        
        return binary_edges, instance_mask
        
    except Exception as e:
        print(f"  Error in process_single_image_gpu: {e}")
        import traceback
        traceback.print_exc()
        torch.cuda.empty_cache()
        return None, None

def process_folders_gpu(sam_folder, point_folder, output_folder, target_size=256):
    """Process all images in folders with GPU acceleration."""
    # Check if folders exist
    if not os.path.exists(sam_folder):
        print(f"Error: SAM folder '{sam_folder}' does not exist!")
        return
    
    if not os.path.exists(point_folder):
        print(f"Error: Point folder '{point_folder}' does not exist!")
        return
    
    # Create output directory
    os.makedirs(output_folder, exist_ok=True)
    print(f"Output folder: {output_folder}")
    print(f"Target size: {target_size}x{target_size}")
    print(f"Ignoring background color: {BACKGROUND_COLOR}")
    
    # Get file lists
    sam_files = sorted([f for f in os.listdir(sam_folder) if f.endswith(('.png', '.jpg', '.jpeg'))])
    point_files = sorted([f for f in os.listdir(point_folder) if f.endswith(('.png', '.jpg', '.jpeg'))])
    
    print(f"Found {len(sam_files)} SAM files and {len(point_files)} point annotation files")
    
    if not sam_files:
        print("No SAM files found!")
        return
    
    if not point_files:
        print("No point annotation files found!")
        return
    
    successful = 0
    failed = 0
    skipped = 0
    
    start_time = time.time()
    
    # Process each file
    for i, sam_file in enumerate(sam_files):
        print(f"\nProcessing {i+1}/{len(sam_files)}: {sam_file}")
        
        point_file = sam_file
        if point_file not in point_files:
            print(f"  Warning: No point annotation found for {sam_file}, skipping...")
            skipped += 1
            continue
        
        sam_path = os.path.join(sam_folder, sam_file)
        point_path = os.path.join(point_folder, point_file)
        
        # Check if files exist
        if not os.path.exists(sam_path):
            print(f"  Error: SAM file {sam_path} does not exist!")
            failed += 1
            continue
            
        if not os.path.exists(point_path):
            print(f"  Error: Point file {point_path} does not exist!")
            failed += 1
            continue
        
        try:
            result = process_single_image_gpu(sam_path, point_path, output_folder, target_size)
            if result[0] is not None:
                successful += 1
            else:
                failed += 1
        except Exception as e:
            print(f"  Error processing {sam_file}: {e}")
            failed += 1
            
        # Print progress every 10 images
        if (i + 1) % 10 == 0:
            elapsed = time.time() - start_time
            images_per_second = (i + 1) / elapsed
            print(f"  Processed {i+1}/{len(sam_files)} images in {elapsed:.1f}s ({images_per_second:.1f} img/s)")
    
    end_time = time.time()
    total_time = end_time - start_time
    
    print(f"\n{'='*50}")
    print(f"GPU PROCESSING COMPLETE!")
    print(f"{'='*50}")
    print(f"Total time: {total_time:.2f} seconds")
    print(f"Images processed: {len(sam_files)}")
    print(f"Successful: {successful}")
    print(f"Failed: {failed}")
    print(f"Skipped: {skipped}")
    print(f"Average speed: {len(sam_files)/total_time:.1f} images/second")
    print(f"Output folder: {output_folder}")
    print(f"Background color ignored: {BACKGROUND_COLOR}")
    
    # Show output files
    output_files = [f for f in os.listdir(output_folder) if f.endswith('_edges.png')]
    if output_files:
        print(f"\nGenerated {len(output_files)} edge maps:")
        for f in output_files[:5]:
            print(f"  {f}")
        if len(output_files) > 5:
            print(f"  ... and {len(output_files) - 5} more")
    else:
        print(f"\nNo output files found in {output_folder}")

# === JUPYTER NOTEBOOK USAGE ===
# Set your paths here (USE ABSOLUTE PATHS):
sam_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_outputs"  # CHANGE THIS
point_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_point_masks_all_regions_1"  # CHANGE THIS  
output_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_outputs_edge_maps" # CHANGE THIS
target_size = 256  # Point annotation size

print("Starting GPU processing...")
print(f"SAM folder: {sam_folder}")
print(f"Point folder: {point_folder}")
print(f"Output folder: {output_folder}")
print(f"Resizing SAM outputs to: {target_size}x{target_size}")
print(f"Ignoring background color: {BACKGROUND_COLOR}")

# Run the GPU processing
process_folders_gpu(sam_folder, point_folder, output_folder, target_size)

print("\nGPU processing completed!")

Using device: cuda
GPU: NVIDIA RTX A5000
GPU Memory: 23.7 GB
Starting GPU processing...
SAM folder: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_outputs
Point folder: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_point_masks_all_regions_1
Output folder: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_outputs_edge_maps
Resizing SAM outputs to: 256x256
Ignoring background color: (100, 100, 100)
Output folder: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_outputs_edge_maps
Target size: 256x256
Ignoring background color: (100, 100, 100)
Found 630 SAM files and 630 point annotation files

Processing 1/630: 1.png
  Original SAM size: (616, 616), Point size: (256, 256)
  Resizing SAM from (616, 616) to (256, 256)
  Found 2 color groups in point annotations (excluding background)
  Colors found: [(0, 255, 0), (160, 160, 164)]
    Color (0, 255, 0): merged 5 segments into instance 1
    Color (160, 160, 164): merged 5 segments into instance 2
  Created 2 ins

In [5]:
import os
import numpy as np
import torch
import cv2
from PIL import Image
import time

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Define background color to ignore
BACKGROUND_COLOR = (100, 100, 100)  # Gray background

def rgb_to_label_mask_gpu(rgb_image):
    """Convert RGB segmentation image to label mask on GPU."""
    rgb_tensor = torch.from_numpy(np.array(rgb_image)).to(device)
    
    if rgb_tensor.dim() == 3 and rgb_tensor.shape[2] == 3:
        integer_image = (rgb_tensor[:, :, 0].long() << 16) + \
                       (rgb_tensor[:, :, 1].long() << 8) + \
                       (rgb_tensor[:, :, 2].long())
        
        unique_colors = torch.unique(integer_image)
        label_mask = torch.zeros_like(integer_image, dtype=torch.int32, device=device)
        
        for label_id, color in enumerate(unique_colors, start=1):
            label_mask[integer_image == color] = label_id
            
        return label_mask.cpu().numpy()
    else:
        return rgb_tensor.cpu().numpy().astype(np.int32)

def get_color_groups_gpu(point_rgb_image):
    """Extract point groups by color from point annotation image using GPU, ignoring background."""
    point_tensor = torch.from_numpy(np.array(point_rgb_image)).to(device)
    color_groups = {}
    
    if point_tensor.dim() == 3 and point_tensor.shape[2] == 3:
        unique_colors = torch.unique(point_tensor.view(-1, 3), dim=0)
        
        bg_color = torch.tensor(BACKGROUND_COLOR, device=device, dtype=point_tensor.dtype)
        mask = ~torch.all(unique_colors == bg_color, dim=1)
        unique_colors = unique_colors[mask]
        
        for color in unique_colors:
            r, g, b = color
            mask = (point_tensor[:, :, 0] == r) & \
                   (point_tensor[:, :, 1] == g) & \
                   (point_tensor[:, :, 2] == b)
            color_groups[tuple(color.cpu().numpy())] = mask.cpu().numpy().astype(np.uint8)
            
    return color_groups

def create_dense_like_edges(instance_mask):
    """
    Create clean, continuous edges similar to dense mask boundaries.
    This mimics what you'd get from boundaries of a proper dense segmentation.
    """
    # Create a binary mask from the instance mask
    binary_mask = (instance_mask > 0).astype(np.uint8) * 255
    
    # Apply morphological operations to create smoother, more continuous regions
    kernel = np.ones((5, 5), np.uint8)
    smoothed_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel)
    smoothed_mask = cv2.morphologyEx(smoothed_mask, cv2.MORPH_OPEN, kernel)
    
    # Apply Gaussian blur to further smooth the edges
    blurred = cv2.GaussianBlur(smoothed_mask, (7, 7), 0)
    
    # Find external contours - this gives the clean outer boundaries
    contours, _ = cv2.findContours(blurred, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Create a clean edge map
    edge_map = np.zeros_like(binary_mask)
    
    for contour in contours:
        # Only draw contours for reasonably sized objects
        if cv2.contourArea(contour) > 50:
            # Draw the contour with smoothing
            epsilon = 0.005 * cv2.arcLength(contour, True)
            approx = cv2.approxPolyDP(contour, epsilon, True)
            cv2.drawContours(edge_map, [approx], -1, 255, 1)
    
    # Optional: Thin the edges to 1-pixel width
    edge_map = cv2.ximgproc.thinning(edge_map)
    
    return edge_map

def process_single_image_gpu(sam_image_path, point_image_path, output_dir, target_size=256):
    """Process a single image pair to create dense-like edges."""
    try:
        # Load images
        sam_image = Image.open(sam_image_path).convert('RGB')
        point_image = Image.open(point_image_path).convert('RGB')
        
        print(f"  Original SAM size: {sam_image.size}, Point size: {point_image.size}")
        
        # Resize SAM image to match point annotation size
        if sam_image.size != (target_size, target_size):
            print(f"  Resizing SAM from {sam_image.size} to ({target_size}, {target_size})")
            sam_image = sam_image.resize((target_size, target_size), Image.NEAREST)
        
        # Convert SAM image to label mask on GPU
        sam_mask = rgb_to_label_mask_gpu(sam_image)
        color_groups = get_color_groups_gpu(point_image)
        
        print(f"  Found {len(color_groups)} color groups in point annotations")
        
        if not color_groups:
            print(f"  No point annotations found in {point_image_path}")
            return None, None
        
        # Move sam_mask to GPU for faster processing
        sam_mask_tensor = torch.from_numpy(sam_mask).to(device)
        height, width = sam_mask.shape
        
        instance_mask = np.zeros((height, width), dtype=np.int32)
        current_instance_id = 1
        
        for color, point_mask in color_groups.items():
            point_mask_tensor = torch.from_numpy(point_mask).to(device)
            
            masked_sam = sam_mask_tensor[point_mask_tensor > 0]
            segments_with_points = torch.unique(masked_sam)
            segments_with_points = segments_with_points[segments_with_points > 0]
            
            if len(segments_with_points) > 0:
                merge_mask = torch.isin(sam_mask_tensor, segments_with_points)
                instance_mask[merge_mask.cpu().numpy()] = current_instance_id
                current_instance_id += 1
        
        # Create DENSE-LIKE edges (clean, continuous boundaries)
        binary_edges = create_dense_like_edges(instance_mask)
        
        base_name = os.path.splitext(os.path.basename(sam_image_path))[0]
        
        edge_image = Image.fromarray(binary_edges)
        edge_output_path = os.path.join(output_dir, f"{base_name}_edges.png")
        edge_image.save(edge_output_path)
        
        print(f"  Created {current_instance_id-1} instances with dense-like edges")
        
        # Clear GPU memory
        torch.cuda.empty_cache()
        
        return binary_edges, instance_mask
        
    except Exception as e:
        print(f"  Error in process_single_image_gpu: {e}")
        import traceback
        traceback.print_exc()
        torch.cuda.empty_cache()
        return None, None

# The rest of the folder processing code remains the same...
def process_folders_gpu(sam_folder, point_folder, output_folder, target_size=256):
    """Process all images in folders to create dense-like edges."""
    if not os.path.exists(sam_folder):
        print(f"Error: SAM folder '{sam_folder}' does not exist!")
        return
    
    if not os.path.exists(point_folder):
        print(f"Error: Point folder '{point_folder}' does not exist!")
        return
    
    os.makedirs(output_folder, exist_ok=True)
    print(f"Output folder: {output_folder}")
    
    sam_files = sorted([f for f in os.listdir(sam_folder) if f.endswith(('.png', '.jpg', '.jpeg'))])
    point_files = sorted([f for f in os.listdir(point_folder) if f.endswith(('.png', '.jpg', '.jpeg'))])
    
    print(f"Found {len(sam_files)} SAM files and {len(point_files)} point annotation files")
    
    successful = 0
    failed = 0
    skipped = 0
    
    start_time = time.time()
    
    for i, sam_file in enumerate(sam_files):
        print(f"\nProcessing {i+1}/{len(sam_files)}: {sam_file}")
        
        point_file = sam_file
        if point_file not in point_files:
            print(f"  Warning: No point annotation found for {sam_file}, skipping...")
            skipped += 1
            continue
        
        sam_path = os.path.join(sam_folder, sam_file)
        point_path = os.path.join(point_folder, point_file)
        
        if not os.path.exists(sam_path):
            print(f"  Error: SAM file {sam_path} does not exist!")
            failed += 1
            continue
            
        if not os.path.exists(point_path):
            print(f"  Error: Point file {point_path} does not exist!")
            failed += 1
            continue
        
        try:
            result = process_single_image_gpu(sam_path, point_path, output_folder, target_size)
            if result[0] is not None:
                successful += 1
            else:
                failed += 1
        except Exception as e:
            print(f"  Error processing {sam_file}: {e}")
            failed += 1
            
        if (i + 1) % 10 == 0:
            elapsed = time.time() - start_time
            print(f"  Processed {i+1}/{len(sam_files)} images in {elapsed:.1f}s")
    
    end_time = time.time()
    total_time = end_time - start_time
    
    print(f"\n{'='*50}")
    print(f"DENSE-LIKE EDGE PROCESSING COMPLETE!")
    print(f"{'='*50}")
    print(f"Total time: {total_time:.2f} seconds")
    print(f"Successful: {successful}")
    print(f"Failed: {failed}")
    print(f"Skipped: {skipped}")

# === USAGE ===
sam_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_outputs"  # CHANGE THIS
point_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_point_masks_all_regions_1"  # CHANGE THIS  
output_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_outputs_edge_maps" # CHANGE THIS
target_size = 256

process_folders_gpu(sam_folder, point_folder, output_folder, target_size)

Using device: cuda
Output folder: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_outputs_edge_maps
Found 630 SAM files and 630 point annotation files

Processing 1/630: 1.png
  Original SAM size: (616, 616), Point size: (256, 256)
  Resizing SAM from (616, 616) to (256, 256)
  Found 2 color groups in point annotations
  Error in process_single_image_gpu: module 'cv2' has no attribute 'ximgproc'

Processing 2/630: 10.png
  Original SAM size: (616, 616), Point size: (256, 256)
  Resizing SAM from (616, 616) to (256, 256)


Traceback (most recent call last):
  File "/tmp/ipykernel_767164/3275452984.py", line 134, in process_single_image_gpu
    binary_edges = create_dense_like_edges(instance_mask)
  File "/tmp/ipykernel_767164/3275452984.py", line 86, in create_dense_like_edges
    edge_map = cv2.ximgproc.thinning(edge_map)
AttributeError: module 'cv2' has no attribute 'ximgproc'


  Found 1 color groups in point annotations
  Error in process_single_image_gpu: module 'cv2' has no attribute 'ximgproc'

Processing 3/630: 100.png
  Original SAM size: (616, 616), Point size: (256, 256)
  Resizing SAM from (616, 616) to (256, 256)


Traceback (most recent call last):
  File "/tmp/ipykernel_767164/3275452984.py", line 134, in process_single_image_gpu
    binary_edges = create_dense_like_edges(instance_mask)
  File "/tmp/ipykernel_767164/3275452984.py", line 86, in create_dense_like_edges
    edge_map = cv2.ximgproc.thinning(edge_map)
AttributeError: module 'cv2' has no attribute 'ximgproc'


  Found 2 color groups in point annotations
  Error in process_single_image_gpu: module 'cv2' has no attribute 'ximgproc'

Processing 4/630: 101.png
  Original SAM size: (616, 616), Point size: (256, 256)
  Resizing SAM from (616, 616) to (256, 256)
  Found 2 color groups in point annotations
  Error in process_single_image_gpu: module 'cv2' has no attribute 'ximgproc'

Processing 5/630: 102.png
  Original SAM size: (616, 616), Point size: (256, 256)
  Resizing SAM from (616, 616) to (256, 256)


...

Traceback (most recent call last):
  File "/tmp/ipykernel_767164/3275452984.py", line 134, in process_single_image_gpu
    binary_edges = create_dense_like_edges(instance_mask)
  File "/tmp/ipykernel_767164/3275452984.py", line 86, in create_dense_like_edges
    edge_map = cv2.ximgproc.thinning(edge_map)
AttributeError: module 'cv2' has no attribute 'ximgproc'
Traceback (most recent call last):
  File "/tmp/ipykernel_767164/3275452984.py", line 134, in process_single_image_gpu
    binary_edges = create_dense_like_edges(instance_mask)
  File "/tmp/ipykernel_767164/3275452984.py", line 86, in create_dense_like_edges
    edge_map = cv2.ximgproc.thinning(edge_map)
AttributeError: module 'cv2' has no attribute 'ximgproc'


  Found 2 color groups in point annotations
  Error in process_single_image_gpu: module 'cv2' has no attribute 'ximgproc'

Processing 629/630: 98.png
  Original SAM size: (616, 616), Point size: (256, 256)
  Resizing SAM from (616, 616) to (256, 256)
  Found 2 color groups in point annotations
  Error in process_single_image_gpu: module 'cv2' has no attribute 'ximgproc'

Processing 630/630: 99.png
  Original SAM size: (616, 616), Point size: (256, 256)
  Resizing SAM from (616, 616) to (256, 256)


Traceback (most recent call last):
  File "/tmp/ipykernel_767164/3275452984.py", line 134, in process_single_image_gpu
    binary_edges = create_dense_like_edges(instance_mask)
  File "/tmp/ipykernel_767164/3275452984.py", line 86, in create_dense_like_edges
    edge_map = cv2.ximgproc.thinning(edge_map)
AttributeError: module 'cv2' has no attribute 'ximgproc'
Traceback (most recent call last):
  File "/tmp/ipykernel_767164/3275452984.py", line 134, in process_single_image_gpu
    binary_edges = create_dense_like_edges(instance_mask)
  File "/tmp/ipykernel_767164/3275452984.py", line 86, in create_dense_like_edges
    edge_map = cv2.ximgproc.thinning(edge_map)
AttributeError: module 'cv2' has no attribute 'ximgproc'


  Found 2 color groups in point annotations
  Error in process_single_image_gpu: module 'cv2' has no attribute 'ximgproc'
  Processed 630/630 images in 379.0s

DENSE-LIKE EDGE PROCESSING COMPLETE!
Total time: 378.97 seconds
Successful: 0
Failed: 630
Skipped: 0


Traceback (most recent call last):
  File "/tmp/ipykernel_767164/3275452984.py", line 134, in process_single_image_gpu
    binary_edges = create_dense_like_edges(instance_mask)
  File "/tmp/ipykernel_767164/3275452984.py", line 86, in create_dense_like_edges
    edge_map = cv2.ximgproc.thinning(edge_map)
AttributeError: module 'cv2' has no attribute 'ximgproc'


In [1]:
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
import cv2
import os
import numpy as np
from tqdm import tqdm

def extract_and_merge_object_edges(input_folder, output_folder, model_path):
    """
    Extract edges of all objects and merge them into a single final image
    """
    # Initialize SAM
    print("🚀 Initializing SAM model...")
    sam = sam_model_registry["vit_h"](checkpoint=model_path)
    sam.to(device="cuda")
    
    mask_generator = SamAutomaticMaskGenerator(
        model=sam,
        points_per_side=32,
        pred_iou_thresh=0.88,
        stability_score_thresh=0.92,
        min_mask_region_area=100,
    )
    
    # Create output folder
    os.makedirs(output_folder, exist_ok=True)
    
    # Get all image files
    image_files = [f for f in os.listdir(input_folder) 
                  if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
    
    print(f"🔍 Found {len(image_files)} images to process")
    
    # Process each image
    for img_file in tqdm(image_files, desc="Processing images"):
        try:
            # Read image
            img_path = os.path.join(input_folder, img_file)
            image = cv2.imread(img_path)
            if image is None:
                print(f"❌ Could not read image: {img_file}")
                continue
            
            # Convert to RGB for SAM
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            # Generate masks
            masks = mask_generator.generate(image_rgb)
            
            print(f"✅ Found {len(masks)} objects in {img_file}!")
            
            # Create a blank white canvas for merged edges
            merged_edges = np.ones_like(image) * 255  # White background
            
            # Process each mask to extract edges and merge directly
            for i, mask_data in enumerate(masks):
                mask = mask_data["segmentation"]
                
                # Convert mask to binary image
                mask_binary = (mask * 255).astype(np.uint8)
                
                # Find contours (edges) of the object
                contours, _ = cv2.findContours(mask_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                
                if contours:
                    # Draw the contour directly on the merged image
                    color = (0, 0, 0)  # Black edges for all objects
                    cv2.drawContours(merged_edges, contours, -1, color, 2)
            
            # Save ONLY the final merged image
            base_name = os.path.splitext(img_file)[0]
            output_path = os.path.join(output_folder, f"{base_name}_merged_edges.jpg")
            cv2.imwrite(output_path, merged_edges)
            
            print(f"🎉 Created merged edges for {img_file}: {len(masks)} objects")
            
        except Exception as e:
            print(f"❌ Error processing {img_file}: {str(e)}")
    
    print(f"✨ Processing complete! Final merged images saved to: {output_folder}")

# Configuration
MODEL_PATH = "/home/iiitdmk-param/Desktop/AA/3.1/Codes/sam_vit_h_4b8939.pth"
INPUT_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_images"
OUTPUT_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_edge_new"

if __name__ == "__main__":
    print("🎯 Extracting and merging object edges into single images...")
    extract_and_merge_object_edges(INPUT_FOLDER, OUTPUT_FOLDER, MODEL_PATH)

🎯 Extracting and merging object edges into single images...
🚀 Initializing SAM model...


/home/iiitdmk-param/AMB/Proposed_3/SAM/segment-anything-main/segment_anything/build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


🔍 Found 630 images to process


Processing images:   0%|                        | 1/630 [00:02<28:51,  2.75s/it]

✅ Found 45 objects in 251.png!
🎉 Created merged edges for 251.png: 45 objects


...

✅ Found 101 objects in 630.png!
🎉 Created merged edges for 630.png: 101 objects


Processing images: 100%|█████████████████████▉| 629/630 [24:57<00:02,  2.35s/it]

✅ Found 25 objects in 67.png!
🎉 Created merged edges for 67.png: 25 objects


Processing images: 100%|██████████████████████| 630/630 [25:00<00:00,  2.38s/it]

✅ Found 16 objects in 263.png!
🎉 Created merged edges for 263.png: 16 objects
✨ Processing complete! Final merged images saved to: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_edge_new


In [2]:
import cv2
import os
import numpy as np
import json
from scipy import ndimage
from tqdm import tqdm

def create_single_merged_mask(sam_output_folder, point_annotations_folder, output_folder):
    """
    Create a single merged mask for each image by merging adjacent SAM regions
    with same-class point annotations
    """
    # Create output folder
    os.makedirs(output_folder, exist_ok=True)
    
    # Get all SAM output directories
    sam_dirs = [d for d in os.listdir(sam_output_folder) 
                if os.path.isdir(os.path.join(sam_output_folder, d))]
    
    print(f"🔍 Found {len(sam_dirs)} SAM output directories to process")
    
    # Process each SAM output directory
    for sam_dir in tqdm(sam_dirs, desc="Processing images"):
        try:
            sam_dir_path = os.path.join(sam_output_folder, sam_dir)
            
            # Load point annotations
            annotation_path = os.path.join(point_annotations_folder, f"{sam_dir}.json")
            
            if not os.path.exists(annotation_path):
                print(f"❌ No annotations found for {sam_dir}")
                continue
            
            with open(annotation_path, 'r') as f:
                point_annotations = json.load(f)
            
            # Load all mask files from SAM output
            mask_files = [f for f in os.listdir(sam_dir_path) 
                         if f.startswith('mask_') and f.endswith('.png')]
            
            if not mask_files:
                print(f"❌ No mask files found in {sam_dir}")
                continue
            
            # Get image shape from first mask
            first_mask_path = os.path.join(sam_dir_path, mask_files[0])
            first_mask = cv2.imread(first_mask_path, cv2.IMREAD_GRAYSCALE)
            image_shape = first_mask.shape
            
            # Load and process all masks
            masks = []
            for mask_file in sorted(mask_files):
                mask_path = os.path.join(sam_dir_path, mask_file)
                mask_img = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if mask_img is not None:
                    mask_binary = (mask_img > 128).astype(bool)
                    masks.append({
                        "segmentation": mask_binary,
                        "filename": mask_file
                    })
            
            # Assign classes to masks based on point annotations
            classified_masks = assign_classes_from_points(masks, point_annotations, image_shape)
            
            # Merge adjacent regions with same class
            merged_regions = merge_adjacent_regions(classified_masks)
            
            # Create single merged mask
            merged_mask = create_single_mask(merged_regions, image_shape)
            
            # Save the single merged mask
            output_path = os.path.join(output_folder, f"{sam_dir}_merged_mask.png")
            cv2.imwrite(output_path, merged_mask)
            
            print(f"✅ Created merged mask for {sam_dir}")
            
        except Exception as e:
            print(f"❌ Error processing {sam_dir}: {str(e)}")
    
    print(f"✨ Processing complete! Single merged masks saved to: {output_folder}")

def assign_classes_from_points(masks, point_annotations, image_shape):
    """Assign classes to masks based on point annotations"""
    classified_masks = []
    
    # Create point location map
    point_map = np.zeros(image_shape[:2], dtype=int)  # 0 means no point
    
    for ann in point_annotations:
        class_id = ann['class_id']
        for point in ann['points']:
            x, y = int(point['x']), int(point['y'])
            if 0 <= x < image_shape[1] and 0 <= y < image_shape[0]:
                point_map[y, x] = class_id
    
    # Assign classes to masks
    for mask_data in masks:
        mask = mask_data["segmentation"]
        
        # Find points inside this mask
        points_in_mask = point_map[mask]
        unique_classes, counts = np.unique(points_in_mask[points_in_mask != 0], return_counts=True)
        
        if len(unique_classes) > 0:
            # Assign the most frequent class in this mask
            assigned_class = unique_classes[np.argmax(counts)]
        else:
            assigned_class = 0  # background (no points)
        
        mask_data['class_id'] = assigned_class
        classified_masks.append(mask_data)
    
    return classified_masks

def merge_adjacent_regions(classified_masks):
    """Merge adjacent regions that have the same class"""
    # Filter out background masks (class 0)
    valid_masks = [m for m in classified_masks if m['class_id'] != 0]
    
    if not valid_masks:
        return classified_masks
    
    # Group masks by class
    class_groups = {}
    for mask_data in valid_masks:
        class_id = mask_data['class_id']
        if class_id not in class_groups:
            class_groups[class_id] = []
        class_groups[class_id].append(mask_data)
    
    merged_regions = []
    
    # Process each class group
    for class_id, masks_in_class in class_groups.items():
        if len(masks_in_class) <= 1:
            # Single mask, no merging needed
            merged_regions.extend(masks_in_class)
            continue
        
        # Create adjacency matrix for this class
        adjacency_matrix = np.zeros((len(masks_in_class), len(masks_in_class)), dtype=bool)
        
        # Check adjacency between all pairs
        for i in range(len(masks_in_class)):
            for j in range(i + 1, len(masks_in_class)):
                if are_masks_adjacent(masks_in_class[i]["segmentation"], masks_in_class[j]["segmentation"]):
                    adjacency_matrix[i, j] = True
                    adjacency_matrix[j, i] = True
        
        # Find connected components (groups of adjacent masks)
        visited = [False] * len(masks_in_class)
        for i in range(len(masks_in_class)):
            if not visited[i]:
                # Start new group
                group = []
                stack = [i]
                visited[i] = True
                
                while stack:
                    current = stack.pop()
                    group.append(current)
                    
                    # Add all adjacent unvisited masks
                    for neighbor in np.where(adjacency_matrix[current])[0]:
                        if not visited[neighbor]:
                            stack.append(neighbor)
                            visited[neighbor] = True
                
                # Merge masks in this group
                if len(group) == 1:
                    merged_regions.append(masks_in_class[group[0]])
                else:
                    merged_mask = merge_mask_group([masks_in_class[idx] for idx in group])
                    merged_mask['class_id'] = class_id
                    merged_regions.append(merged_mask)
    
    return merged_regions

def are_masks_adjacent(mask1, mask2):
    """Check if two masks are adjacent (touching each other)"""
    # Dilate mask1 slightly
    dilated_mask1 = ndimage.binary_dilation(mask1, structure=np.ones((3, 3)))
    
    # Check if mask2 touches dilated mask1
    return np.any(np.logical_and(dilated_mask1, mask2))

def merge_mask_group(mask_group):
    """Merge a group of masks into one"""
    merged_mask = mask_group[0]["segmentation"].copy()
    
    for i in range(1, len(mask_group)):
        merged_mask = np.logical_or(merged_mask, mask_group[i]["segmentation"])
    
    # Create new mask data
    merged_data = {
        "segmentation": merged_mask,
        "area": np.sum(merged_mask)
    }
    
    return merged_data

def create_single_mask(merged_regions, image_shape):
    """Create a single merged mask with class values"""
    # Create empty mask
    final_mask = np.zeros(image_shape, dtype=np.uint8)
    
    # Assign class values to each region
    for region in merged_regions:
        if region['class_id'] != 0:  # Skip background
            final_mask[region["segmentation"]] = region['class_id']
    
    return final_mask

# Configuration
SAM_OUTPUT_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_edge_new"
POINT_ANNOTATIONS_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_point_masks_all_regions_1"
OUTPUT_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_edge_merged"

if __name__ == "__main__":
    print("🎯 Creating single merged masks for each image...")
    create_single_merged_mask(
        SAM_OUTPUT_FOLDER, 
        POINT_ANNOTATIONS_FOLDER, 
        OUTPUT_FOLDER
    )

🎯 Creating single merged masks for each image...
🔍 Found 0 SAM output directories to process


Processing images: 0it [00:00, ?it/s]

✨ Processing complete! Single merged masks saved to: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_edge_merged


In [4]:
import cv2
import os
import numpy as np
from scipy import ndimage
from tqdm import tqdm

# Your class color mapping (BGR format for OpenCV)
CLASS_COLORS = {
    1: (240, 202, 166),   # airplane (BGR)
    2: (0, 128, 128),     # bare soil (BGR)
    3: (128, 0, 0),       # buildings (BGR)
    4: (0, 0, 255),       # cars (BGR)
    5: (0, 128, 0),       # chaparral (BGR)
    6: (0, 0, 128),       # court (BGR)
    7: (233, 233, 255),   # dock (BGR)
    8: (164, 160, 160),   # field (BGR)
    9: (128, 128, 0),     # grass (BGR)
    10: (255, 87, 90),    # mobile home (BGR)
    11: (0, 255, 255),    # pavement (BGR)
    12: (0, 192, 255),    # sand (BGR)
    13: (255, 0, 0),      # sea (BGR)
    14: (92, 0, 255),     # ship (BGR)
    15: (128, 0, 128),    # tanks (BGR)
    16: (0, 255, 0),      # trees (BGR)
    17: (255, 255, 0)     # water (BGR)
}

BACKGROUND_COLOR = (100, 100, 100)  # Background color to ignore

def merge_sam_edges_with_point_guidance(sam_edges_folder, point_masks_folder, output_folder):
    """
    Merge SAM's over-segmented edges using point annotations as guidance
    Points of same color = same object = merge corresponding SAM regions
    Ignore background color (100,100,100)
    """
    # Create output folder
    os.makedirs(output_folder, exist_ok=True)
    
    # Get all point mask files
    point_mask_files = [f for f in os.listdir(point_masks_folder) 
                       if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
    
    print(f"🔍 Found {len(point_mask_files)} point mask files to process")
    
    # Process each point mask
    for point_mask_file in tqdm(point_mask_files, desc="Merging SAM edges"):
        try:
            base_name = os.path.splitext(point_mask_file)[0]
            
            # Load point mask
            point_mask_path = os.path.join(point_masks_folder, point_mask_file)
            point_mask = cv2.imread(point_mask_path)
            
            if point_mask is None:
                print(f"❌ Could not read point mask: {point_mask_file}")
                continue
            
            # Load corresponding SAM edges
            sam_edges_path = os.path.join(sam_edges_folder, f"{base_name}_merged_edges.jpg")
            if not os.path.exists(sam_edges_path):
                # Try different naming patterns
                sam_edges_path = os.path.join(sam_edges_folder, f"{base_name}.png")
                if not os.path.exists(sam_edges_path):
                    print(f"❌ No SAM edges found for {base_name}")
                    continue
            
            sam_edges = cv2.imread(sam_edges_path, cv2.IMREAD_GRAYSCALE)
            if sam_edges is None:
                print(f"❌ Could not read SAM edges for {base_name}")
                continue
            
            # Ensure same size
            if point_mask.shape[:2] != sam_edges.shape:
                point_mask = cv2.resize(point_mask, (sam_edges.shape[1], sam_edges.shape[0]))
            
            # Create background mask to ignore (100,100,100) pixels
            background_mask = create_background_mask(point_mask)
            
            # Extract points from point mask (ignoring background)
            points_by_class = extract_points_from_mask(point_mask, background_mask)
            
            # Convert SAM edges to binary mask (edges = black)
            sam_binary = (sam_edges < 128).astype(np.uint8)
            
            # Find connected components in SAM edges
            num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(sam_binary, connectivity=8)
            
            # Create SAM regions
            sam_regions = []
            for i in range(1, num_labels):  # Skip background (label 0)
                region_mask = (labels == i).astype(bool)
                sam_regions.append({
                    "segmentation": region_mask,
                    "area": np.sum(region_mask),
                    "label": i
                })
            
            print(f"📊 {base_name}: SAM found {len(sam_regions)} regions, points for {len(points_by_class)} classes")
            
            # Assign classes to SAM regions based on points
            classified_regions = assign_classes_to_sam_regions(sam_regions, points_by_class, sam_edges.shape)
            
            # Merge SAM regions that belong to the same object (same class points)
            merged_mask = merge_regions_by_class(classified_regions, sam_edges.shape)
            
            # Convert merged mask to edges (white background, black edges)
            final_edges = create_clean_edges(merged_mask)
            
            # Save result
            output_path = os.path.join(output_folder, f"{base_name}.png")
            cv2.imwrite(output_path, final_edges)
            
            print(f"✅ Merged {base_name}: {len(sam_regions)} → {len(points_by_class)} objects")
            
        except Exception as e:
            print(f"❌ Error processing {point_mask_file}: {str(e)}")
            import traceback
            traceback.print_exc()
    
    print(f"✨ Processing complete! Merged edge masks saved to: {output_folder}")

def create_background_mask(point_mask):
    """Create mask for background color (100,100,100) to ignore"""
    lower_bound = np.array([95, 95, 95])  # Slight tolerance
    upper_bound = np.array([105, 105, 105])
    background_mask = cv2.inRange(point_mask, lower_bound, upper_bound)
    return background_mask > 0

def extract_points_from_mask(point_mask, background_mask):
    """
    Extract colored points from point annotation mask
    Ignore points that fall on background color (100,100,100)
    Returns: dict {class_id: list_of_points}
    """
    points_by_class = {}
    
    for class_id, color_bgr in CLASS_COLORS.items():
        # Create mask for this specific color (with tolerance)
        lower_bound = np.array([max(0, c-15) for c in color_bgr])
        upper_bound = np.array([min(255, c+15) for c in color_bgr])
        
        color_mask = cv2.inRange(point_mask, lower_bound, upper_bound)
        
        # Remove points that are on background
        color_mask[background_mask] = 0
        
        # Find contours (points)
        contours, _ = cv2.findContours(color_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        points = []
        for contour in contours:
            # Filter by size to get only the annotation points
            if 3 < cv2.contourArea(contour) < 100:  # Point annotation size range
                M = cv2.moments(contour)
                if M["m00"] > 0:
                    cx = int(M["m10"] / M["m00"])
                    cy = int(M["m01"] / M["m00"])
                    # Check if this point is not on background
                    if not background_mask[cy, cx]:
                        points.append((cx, cy))
        
        if points:
            points_by_class[class_id] = points
            print(f"   Class {class_id}: {len(points)} points")
    
    return points_by_class

def assign_classes_to_sam_regions(sam_regions, points_by_class, image_shape):
    """
    Assign classes to SAM regions based on which points they contain
    """
    classified_regions = []
    
    # Create point location map
    point_map = np.zeros(image_shape[:2], dtype=int)  # 0 = no point
    
    for class_id, points in points_by_class.items():
        for (x, y) in points:
            if 0 <= x < image_shape[1] and 0 <= y < image_shape[0]:
                point_map[y, x] = class_id
    
    # Assign classes to SAM regions
    for region in sam_regions:
        mask = region["segmentation"]
        
        # Find points inside this region
        points_in_region = point_map[mask]
        unique_classes, counts = np.unique(points_in_region[points_in_region != 0], return_counts=True)
        
        if len(unique_classes) > 0:
            # Assign the class with most points in this region
            assigned_class = unique_classes[np.argmax(counts)]
            confidence = np.max(counts) / len(points_in_region[points_in_region != 0])
        else:
            assigned_class = 0  # No points in this region
            confidence = 0
        
        region['class_id'] = assigned_class
        region['confidence'] = confidence
        region['point_count'] = np.sum(points_in_region != 0)
        
        classified_regions.append(region)
    
    return classified_regions

def merge_regions_by_class(classified_regions, image_shape):
    """
    Merge SAM regions that belong to the same class (same object)
    """
    # Group regions by class (ignore class 0 - no points)
    regions_by_class = {}
    for region in classified_regions:
        if region['class_id'] != 0:
            class_id = region['class_id']
            if class_id not in regions_by_class:
                regions_by_class[class_id] = []
            regions_by_class[class_id].append(region)
    
    # Create merged mask
    merged_mask = np.zeros(image_shape, dtype=np.uint8)
    
    # For each class, merge all regions that belong to it
    for class_id, regions in regions_by_class.items():
        if not regions:
            continue
        
        # Create combined mask for this class
        class_mask = np.zeros(image_shape, dtype=bool)
        for region in regions:
            class_mask = np.logical_or(class_mask, region["segmentation"])
        
        # Assign class ID to the merged region
        merged_mask[class_mask] = class_id
    
    return merged_mask

def create_clean_edges(merged_mask):
    """
    Convert merged class mask to clean edges (white background, black edges)
    """
    # Create white background
    edges = np.ones_like(merged_mask) * 255
    
    # For each class, find edges
    for class_id in np.unique(merged_mask):
        if class_id == 0:  # Skip background
            continue
        
        # Create binary mask for this class
        class_binary = (merged_mask == class_id).astype(np.uint8) * 255
        
        # Find contours
        contours, _ = cv2.findContours(class_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # Draw edges in black
        cv2.drawContours(edges, contours, -1, 0, 2)
    
    return edges

# Configuration
SAM_EDGES_FOLDER =  "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_edge_new"
POINT_MASKS_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_point_masks_all_regions_1"
OUTPUT_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_edge_merged"

if __name__ == "__main__":
    print("🎯 Merging SAM edges using point annotation guidance...")
    print("💡 Strategy: Same color points = same object = merge corresponding SAM regions")
    print("🚫 Ignoring background color (100,100,100)")
    merge_sam_edges_with_point_guidance(
        SAM_EDGES_FOLDER, 
        POINT_MASKS_FOLDER, 
        OUTPUT_FOLDER
    )

🎯 Merging SAM edges using point annotation guidance...
💡 Strategy: Same color points = same object = merge corresponding SAM regions
🚫 Ignoring background color (100,100,100)
🔍 Found 630 point mask files to process


Merging SAM edges:   2%|▎                      | 10/630 [00:00<00:06, 95.83it/s]

📊 251: SAM found 12 regions, points for 0 classes
✅ Merged 251: 12 → 0 objects
📊 542: SAM found 7 regions, points for 0 classes
✅ Merged 542: 7 → 0 objects
📊 8: SAM found 1 regions, points for 0 classes
✅ Merged 8: 1 → 0 objects
📊 233: SAM found 14 regions, points for 0 classes
✅ Merged 233: 14 → 0 objects
📊 611: SAM found 16 regions, points for 0 classes
✅ Merged 611: 16 → 0 objects
📊 302: SAM found 10 regions, points for 0 classes
✅ Merged 302: 10 → 0 objects
📊 73: SAM found 5 regions, points for 0 classes
✅ Merged 73: 5 → 0 objects
📊 424: SAM found 14 regions, points for 0 classes
✅ Merged 424: 14 → 0 objects
📊 103: SAM found 9 regions, points for 0 classes
✅ Merged 103: 9 → 0 objects
📊 247: SAM found 12 regions, points for 0 classes
✅ Merged 247: 12 → 0 objects
📊 180: SAM found 117 regions, points for 0 classes
✅ Merged 180: 117 → 0 objects
📊 583: SAM found 7 regions, points for 0 classes
✅ Merged 583: 7 → 0 objects
📊 264: SAM found 5 regions, points for 0 classes
✅ Merged 264: 5 →

Merging SAM edges:   4%|▊                     | 24/630 [00:00<00:05, 119.78it/s]

📊 609: SAM found 9 regions, points for 0 classes
✅ Merged 609: 9 → 0 objects
📊 11: SAM found 2 regions, points for 0 classes
✅ Merged 11: 2 → 0 objects


...

📊 183: SAM found 4 regions, points for 0 classes
✅ Merged 183: 4 → 0 objects
📊 599: SAM found 16 regions, points for 0 classes
✅ Merged 599: 16 → 0 objects
📊 36: SAM found 7 regions, points for 0 classes
✅ Merged 36: 7 → 0 objects
📊 427: SAM found 8 regions, points for 0 classes
✅ Merged 427: 8 → 0 objects
📊 391: SAM found 3 regions, points for 0 classes
✅ Merged 391: 3 → 0 objects
📊 297: SAM found 8 regions, points for 0 classes
✅ Merged 297: 8 → 0 objects
📊 429: SAM found 2 regions, points for 0 classes
✅ Merged 429: 2 → 0 objects
📊 62: SAM found 1 regions, points for 0 classes
✅ Merged 62: 1 → 0 objects
📊 12: SAM found 1 regions, points for 0 classes
✅ Merged 12: 1 → 0 objects
📊 403: SAM found 4 regions, points for 0 classes
✅ Merged 403: 4 → 0 objects
📊 342: SAM found 8 regions, points for 0 classes
✅ Merged 342: 8 → 0 objects
📊 124: SAM found 17 regions, points for 0 classes
✅ Merged 124: 17 → 0 objects
📊 315: SAM found 2 regions, points for 0 classes
✅ Merged 315: 2 → 0 objects
📊

Merging SAM edges:  95%|███████████████████▊ | 596/630 [00:04<00:00, 158.60it/s]

📊 358: SAM found 14 regions, points for 0 classes
✅ Merged 358: 14 → 0 objects
📊 350: SAM found 28 regions, points for 0 classes
✅ Merged 350: 28 → 0 objects
📊 354: SAM found 9 regions, points for 0 classes
✅ Merged 354: 9 → 0 objects
📊 444: SAM found 13 regions, points for 0 classes
✅ Merged 444: 13 → 0 objects
📊 64: SAM found 2 regions, points for 0 classes
✅ Merged 64: 2 → 0 objects
📊 336: SAM found 10 regions, points for 0 classes
✅ Merged 336: 10 → 0 objects
📊 471: SAM found 9 regions, points for 0 classes
✅ Merged 471: 9 → 0 objects
📊 608: SAM found 10 regions, points for 0 classes
✅ Merged 608: 10 → 0 objects
📊 313: SAM found 3 regions, points for 0 classes
✅ Merged 313: 3 → 0 objects
📊 557: SAM found 9 regions, points for 0 classes
✅ Merged 557: 9 → 0 objects
📊 193: SAM found 2 regions, points for 0 classes
✅ Merged 193: 2 → 0 objects
📊 431: SAM found 5 regions, points for 0 classes
✅ Merged 431: 5 → 0 objects
📊 324: SAM found 1 regions, points for 0 classes
✅ Merged 324: 1 → 0

Merging SAM edges: 100%|█████████████████████| 630/630 [00:04<00:00, 149.36it/s]

📊 266: SAM found 10 regions, points for 0 classes
✅ Merged 266: 10 → 0 objects
📊 406: SAM found 2 regions, points for 0 classes
✅ Merged 406: 2 → 0 objects
📊 402: SAM found 12 regions, points for 0 classes
✅ Merged 402: 12 → 0 objects
📊 18: SAM found 1 regions, points for 0 classes
✅ Merged 18: 1 → 0 objects
📊 201: SAM found 17 regions, points for 0 classes
✅ Merged 201: 17 → 0 objects
📊 95: SAM found 3 regions, points for 0 classes
✅ Merged 95: 3 → 0 objects
📊 316: SAM found 1 regions, points for 0 classes
✅ Merged 316: 1 → 0 objects
📊 396: SAM found 2 regions, points for 0 classes
✅ Merged 396: 2 → 0 objects
📊 442: SAM found 11 regions, points for 0 classes
✅ Merged 442: 11 → 0 objects
📊 446: SAM found 17 regions, points for 0 classes
✅ Merged 446: 17 → 0 objects
📊 561: SAM found 13 regions, points for 0 classes
✅ Merged 561: 13 → 0 objects
📊 308: SAM found 1 regions, points for 0 classes
✅ Merged 308: 1 → 0 objects
📊 523: SAM found 3 regions, points for 0 classes
✅ Merged 523: 3 → 0

In [7]:
import cv2
import os
import numpy as np
from scipy import ndimage
from tqdm import tqdm
import re

# Your class color mapping (BGR format for OpenCV)
CLASS_COLORS = {
    1: (240, 202, 166),   # airplane (BGR)
    2: (0, 128, 128),     # bare soil (BGR)
    3: (128, 0, 0),       # buildings (BGR)
    4: (0, 0, 255),       # cars (BGR)
    5: (0, 128, 0),       # chaparral (BGR)
    6: (0, 0, 128),       # court (BGR)
    7: (233, 233, 255),   # dock (BGR)
    8: (164, 160, 160),   # field (BGR)
    9: (128, 128, 0),     # grass (BGR)
    10: (255, 87, 90),    # mobile home (BGR)
    11: (0, 255, 255),    # pavement (BGR)
    12: (0, 192, 255),    # sand (BGR)
    13: (255, 0, 0),      # sea (BGR)
    14: (92, 0, 255),     # ship (BGR)
    15: (128, 0, 128),    # tanks (BGR)
    16: (0, 255, 0),      # trees (BGR)
    17: (255, 255, 0)     # water (BGR)
}

def extract_number_from_filename(filename):
    """Extract the numeric part from filename"""
    numbers = re.findall(r'\d+', filename)
    return numbers[0] if numbers else None

def merge_sam_edges_with_point_guidance(sam_edges_folder, point_masks_folder, output_folder):
    """
    Merge SAM edges using point annotations as guidance
    Point masks: 1.png, 2.png, 3.png, etc.
    SAM edges: 1_merged_edges.jpg, 2_merged_edges.jpg, etc.
    """
    # Create output folder
    os.makedirs(output_folder, exist_ok=True)
    
    # Get all point mask files
    point_mask_files = [f for f in os.listdir(point_masks_folder) 
                       if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
    
    # Get all SAM edge files
    sam_files = [f for f in os.listdir(sam_edges_folder) 
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')) and 'merged_edges' in f]
    
    print(f"🔍 Found {len(point_mask_files)} point mask files")
    print(f"🔍 Found {len(sam_files)} SAM edge files")
    
    # Create mapping from number to SAM file
    sam_file_map = {}
    for sam_file in sam_files:
        number = extract_number_from_filename(sam_file)
        if number:
            sam_file_map[number] = sam_file
    
    # Process each point mask
    processed_count = 0
    for point_mask_file in tqdm(point_mask_files, desc="Merging SAM edges"):
        try:
            # Extract number from point mask filename
            point_number = extract_number_from_filename(point_mask_file)
            if point_number is None:
                print(f"❌ No number found in filename: {point_mask_file}")
                continue
            
            # Find matching SAM file
            if point_number not in sam_file_map:
                print(f"❌ No SAM file found for point mask {point_mask_file} (number {point_number})")
                continue
            
            sam_file = sam_file_map[point_number]
            
            # Load point mask
            point_mask_path = os.path.join(point_masks_folder, point_mask_file)
            point_mask = cv2.imread(point_mask_path)
            
            # Load SAM edges
            sam_edges_path = os.path.join(sam_edges_folder, sam_file)
            sam_edges = cv2.imread(sam_edges_path, cv2.IMREAD_GRAYSCALE)
            
            if point_mask is None:
                print(f"❌ Could not read point mask: {point_mask_file}")
                continue
            
            if sam_edges is None:
                print(f"❌ Could not read SAM edges: {sam_file}")
                continue
            
            # Ensure same size
            if point_mask.shape[:2] != sam_edges.shape:
                point_mask = cv2.resize(point_mask, (sam_edges.shape[1], sam_edges.shape[0]))
            
            # Extract points from point mask (EXACT color matching)
            points_by_class = extract_points_exact_color(point_mask)
            
            # Check if we found any points
            total_points = sum(len(points) for points in points_by_class.values())
            if total_points == 0:
                print(f"⚠️  No points found in {point_mask_file}")
                continue
            
            # Convert SAM edges to binary regions
            sam_regions = extract_regions_from_edges(sam_edges)
            
            if not sam_regions:
                print(f"⚠️  No regions found in SAM edges for {point_mask_file}")
                continue
            
            print(f"📊 {point_mask_file}: {len(sam_regions)} SAM regions, {total_points} points")
            
            # Assign classes to SAM regions based on points
            classified_regions = assign_classes_to_sam_regions(sam_regions, points_by_class, sam_edges.shape)
            
            # Merge SAM regions that belong to the same object (same class points)
            merged_mask = merge_regions_by_class(classified_regions, sam_edges.shape)
            
            # Convert merged mask to edges (white background, black edges)
            final_edges = create_clean_edges(merged_mask)
            
            # Save result with same name as point mask
            output_path = os.path.join(output_folder, point_mask_file)
            cv2.imwrite(output_path, final_edges)
            
            processed_count += 1
            num_objects = len([cls for cls in points_by_class if points_by_class[cls]])
            print(f"✅ {point_mask_file}: {len(sam_regions)} → {num_objects} objects")
            
        except Exception as e:
            print(f"❌ Error processing {point_mask_file}: {str(e)}")
    
    print(f"✨ Processing complete! {processed_count}/{len(point_mask_files)} files processed")
    print(f"📁 Merged edge masks saved to: {output_folder}")

def extract_regions_from_edges(edges_image):
    """Extract regions from SAM edge image (black edges on white background)"""
    # Convert edges to binary (edges = black, background = white)
    binary = (edges_image < 128).astype(np.uint8)
    
    # Clean up the binary image
    kernel = np.ones((3, 3), np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    
    # Find connected components
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)
    
    regions = []
    for i in range(1, num_labels):  # Skip background (label 0)
        region_mask = (labels == i).astype(bool)
        # Filter out very small regions (noise)
        if np.sum(region_mask) > 10:  # At least 10 pixels
            regions.append({
                "segmentation": region_mask,
                "area": np.sum(region_mask)
            })
    
    return regions

def extract_points_exact_color(point_mask):
    """Extract points using EXACT color matching"""
    points_by_class = {}
    point_mask_np = np.array(point_mask)
    
    for class_id, exact_color in CLASS_COLORS.items():
        # Find pixels that exactly match this color
        color_match = np.all(point_mask_np == exact_color, axis=-1)
        y_coords, x_coords = np.where(color_match)
        
        points = [(x, y) for x, y in zip(x_coords, y_coords)]
        points_by_class[class_id] = points
        
        if points:
            print(f"   Class {class_id}: {len(points)} points")
    
    return points_by_class

def assign_classes_to_sam_regions(sam_regions, points_by_class, image_shape):
    """Assign classes to SAM regions based on which points they contain"""
    classified_regions = []
    point_map = np.zeros(image_shape[:2], dtype=int)
    
    for class_id, points in points_by_class.items():
        for (x, y) in points:
            if 0 <= x < image_shape[1] and 0 <= y < image_shape[0]:
                point_map[y, x] = class_id
    
    for region in sam_regions:
        mask = region["segmentation"]
        points_in_region = point_map[mask]
        unique_classes, counts = np.unique(points_in_region[points_in_region != 0], return_counts=True)
        
        if len(unique_classes) > 0:
            region['class_id'] = unique_classes[np.argmax(counts)]
        else:
            region['class_id'] = 0
        
        classified_regions.append(region)
    
    return classified_regions

def merge_regions_by_class(classified_regions, image_shape):
    """Merge SAM regions that belong to the same class"""
    regions_by_class = {}
    for region in classified_regions:
        if region['class_id'] != 0:
            class_id = region['class_id']
            if class_id not in regions_by_class:
                regions_by_class[class_id] = []
            regions_by_class[class_id].append(region)
    
    merged_mask = np.zeros(image_shape, dtype=np.uint8)
    
    for class_id, regions in regions_by_class.items():
        class_mask = np.zeros(image_shape, dtype=bool)
        for region in regions:
            class_mask = np.logical_or(class_mask, region["segmentation"])
        merged_mask[class_mask] = class_id
    
    return merged_mask

def create_clean_edges(merged_mask):
    """Convert merged class mask to clean edges (white background, black edges)"""
    edges = np.ones_like(merged_mask) * 255
    
    for class_id in np.unique(merged_mask):
        if class_id == 0:
            continue
        
        class_binary = (merged_mask == class_id).astype(np.uint8) * 255
        contours, _ = cv2.findContours(class_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if contours:
            cv2.drawContours(edges, contours, -1, 0, 2)
    
    return edges

# Configuration
SAM_EDGES_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_edge_new"
POINT_MASKS_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_point_masks_all_regions_1"
OUTPUT_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_edge_merged"

if __name__ == "__main__":
    print("🎯 Merging SAM edges using point annotation guidance...")
    print("📁 Point masks: 1.png, 2.png, 3.png, etc.")
    print("📁 SAM edges: 1_merged_edges.jpg, 2_merged_edges.jpg, etc.")
    
    merge_sam_edges_with_point_guidance(
        SAM_EDGES_FOLDER, 
        POINT_MASKS_FOLDER, 
        OUTPUT_FOLDER
    )

🎯 Merging SAM edges using point annotation guidance...
📁 Point masks: 1.png, 2.png, 3.png, etc.
📁 SAM edges: 1_merged_edges.jpg, 2_merged_edges.jpg, etc.
🔍 Found 630 point mask files
🔍 Found 630 SAM edge files


Merging SAM edges:   0%|                        | 3/630 [00:00<00:21, 28.77it/s]

   Class 4: 70 points
   Class 9: 10 points
   Class 11: 15 points
   Class 16: 25 points
📊 251.png: 11 SAM regions, 120 points
✅ 251.png: 11 → 4 objects
   Class 2: 20 points
   Class 3: 10 points
   Class 9: 15 points
   Class 11: 10 points
   Class 16: 25 points
📊 542.png: 4 SAM regions, 80 points
✅ 542.png: 4 → 5 objects
   Class 8: 5 points
📊 8.png: 1 SAM regions, 5 points
✅ 8.png: 1 → 1 objects
   Class 16: 5 points
📊 233.png: 10 SAM regions, 5 points
✅ 233.png: 10 → 1 objects
   Class 3: 15 points
   Class 6: 5 points
   Class 11: 5 points
📊 611.png: 11 SAM regions, 25 points
✅ 611.png: 11 → 3 objects
   Class 7: 10 points


Merging SAM edges:   1%|▎                       | 9/630 [00:00<00:22, 27.36it/s]

   Class 17: 95 points
📊 302.png: 3 SAM regions, 105 points
✅ 302.png: 3 → 2 objects
   Class 2: 5 points
   Class 3: 40 points
   Class 9: 15 points
📊 73.png: 3 SAM regions, 60 points
✅ 73.png: 3 → 3 objects
   Class 2: 25 points
   Class 4: 170 points
   Class 11: 5 points
📊 424.png: 8 SAM regions, 200 points
✅ 424.png: 8 → 3 objects
   Class 12: 5 points
   Class 13: 5 points
📊 103.png: 9 SAM regions, 10 points
✅ 103.png: 9 → 2 objects
   Class 2: 10 points
   Class 4: 65 points
   Class 9: 10 points
   Class 11: 10 points
   Class 16: 5 points
📊 247.png: 9 SAM regions, 100 points
✅ 247.png: 9 → 5 objects
   Class 2: 100 points
   Class 5: 494 points
📊 180.png: 50 SAM regions, 594 points
✅ 180.png: 50 → 2 objects
   Class 3: 5 points


...

   Class 2: 5 points
   Class 3: 10 points
   Class 4: 10 points
   Class 9: 20 points
   Class 11: 15 points
📊 561.png: 12 SAM regions, 60 points
✅ 561.png: 12 → 5 objects
   Class 7: 10 points
   Class 17: 20 points
📊 308.png: 1 SAM regions, 30 points
✅ 308.png: 1 → 2 objects
   Class 9: 25 points
   Class 11: 10 points
📊 523.png: 2 SAM regions, 35 points
✅ 523.png: 2 → 2 objects
   Class 16: 5 points
📊 217.png: 12 SAM regions, 5 points
✅ 217.png: 12 → 1 objects
   Class 12: 5 points
   Class 13: 5 points
   Class 16: 15 points
📊 111.png: 1 SAM regions, 25 points
✅ 111.png: 1 → 3 objects
   Class 2: 15 points
   Class 3: 10 points
   Class 9: 15 points
   Class 11: 5 points
   Class 16: 20 points
📊 560.png: 11 SAM regions, 65 points
✅ 560.png: 11 → 5 objects


Merging SAM edges: 100%|█████████████████████▉| 627/630 [00:21<00:00, 29.10it/s]

   Class 9: 30 points
   Class 11: 5 points
📊 449.png: 10 SAM regions, 35 points
✅ 449.png: 10 → 2 objects
   Class 3: 25 points
   Class 4: 25 points
   Class 9: 39 points
   Class 11: 10 points
   Class 16: 25 points
📊 369.png: 6 SAM regions, 124 points
✅ 369.png: 6 → 5 objects
   Class 9: 5 points
   Class 12: 10 points
   Class 16: 10 points
   Class 17: 5 points
📊 498.png: 7 SAM regions, 30 points
✅ 498.png: 7 → 4 objects
   Class 7: 10 points
   Class 17: 30 points
📊 318.png: 2 SAM regions, 40 points
✅ 318.png: 2 → 2 objects
   Class 4: 283 points
   Class 11: 26 points
   Class 16: 30 points
📊 476.png: 13 SAM regions, 339 points
✅ 476.png: 13 → 3 objects
   Class 3: 20 points
   Class 4: 30 points
   Class 6: 5 points
   Class 9: 35 points
   Class 11: 10 points
   Class 16: 25 points
📊 630.png: 8 SAM regions, 125 points
✅ 630.png: 8 → 6 objects
   Class 2: 5 points


Merging SAM edges: 100%|██████████████████████| 630/630 [00:22<00:00, 28.56it/s]

   Class 9: 5 points
   Class 11: 5 points
   Class 16: 5 points
📊 67.png: 4 SAM regions, 20 points
✅ 67.png: 4 → 4 objects
   Class 2: 5 points
   Class 4: 30 points
   Class 9: 5 points
   Class 11: 5 points
📊 263.png: 8 SAM regions, 45 points
✅ 263.png: 8 → 4 objects
✨ Processing complete! 630/630 files processed
📁 Merged edge masks saved to: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_edge_merged
